# Gemma 4 Test: Google's Next-Gen Lightweight VLM

This notebook tests **Gemma 4**, evaluating its performance on complex Arabic document parsing.

### 1. Setup

```bash
uv pip install openai pillow
```


In [1]:
import base64
import time
from openai import OpenAI
from PIL import Image
import io
import sys

# Point to the local server (LM Studio / Ollama)
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


img_path = "ollama_test_table.png"
base64_image = encode_image(img_path)

print("Setup complete. Ready to talk to Gemma 4.")

Setup complete. Ready to talk to Gemma 4.


### 2. Run Inference: Document Transcription (STREAMING)

We will use a high-precision prompt to test Gemma 4's visual acuity and Arabic spelling.


In [2]:
print("Requesting streaming inference from Gemma 4...\n")
start_time = time.time()

response = client.chat.completions.create(
    model="gemma-4",  # LM Studio uses the loaded model
    messages=[
        {
            "role": "system",
            "content": "You are a world-class document digitization assistant. Transcribe the image faithfully into Markdown. For tables, use pipe format (| column |). Ensure perfect Arabic spelling and preserve the '...' row markers.",
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Please transcribe this Arabic document exactly as it appears.",
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                },
            ],
        },
    ],
    max_tokens=4096,
    temperature=0,
    stream=True,
)

full_response = ""
print("--- Real-time Output ---")
for chunk in response:
    if chunk.choices[0].delta.content:
        content = chunk.choices[0].delta.content
        print(content, end="", flush=True)
        full_response += content

end_time = time.time()
print(f"\n\n--- Inference Finished in {end_time - start_time:.2f} seconds ---")

Requesting streaming inference from Gemma 4...

--- Real-time Output ---
## «الجناح»

| الرقم الترتيبي | العنوان | النقاط الواجب خصمها |
| :---: | :--- | :---: |
| 01 | ... | |
| 07 | سيارة مركبة تحت الكحول أو تحت تأثير المواد المخدرة | |
| ... | أوتحت تأثير المواد المخدرة | |
| ... | - رفض الخضوع لفحص إليه في المادة 207 أدناه أو لتحققات أو لاختبارات الكشف المنصوص عليها في المادتين 213 و 214 أدناه. | |
| 13 | المسائق الذي وجه الأمر بالتوقف ومن تنفيذه أولم يحترم الأمر بتوقيف المركبة أو رفض سيارة مركبته أو عمل على صفاتها إلى المعجز أو رفض الامتثال لأوامر القانونية الصادرة إليه. | |
| ... | ... | |
| 18 | ... | |

## «المخالفات»

| الرقم الترتيبي | المخالفات | النقاط الواجب خصمها |
| :---: | :--- | :---: |
| ... | عدم احترام إجبارة استعمال حزام السلامة. | 19 |
| 30 | - الاستعمال أو التعثث بالاتفاق ممسوكاً بالثناء السياقة أو أي جهاز آخر يقوم بوظائف الهاتف. | 31 |
| 32 | ... | |

--- Inference Finished in 61.73 seconds ---


### 3. Display Rendered Result


In [ ]:
from IPython.display import display, Markdown

display(Markdown(full_response))